# Tutorial 4: Model Checkpoint and Reload with an "everything" Manifest

Save a PyTorch model **and** its optimizer to S3, together with the training config, under a single **`ComplexManifest`** named `everything`; destroy all local state; then get the exact same objects back from that one nickname.

- `laila.addons.torch.Module` / `Optimizer` wrap the torch objects (same call API) so that memorizing them snapshots every tensor as a detached CPU entry, plus a weight-free architecture skeleton for the model and model / parameter references for the optimizer.
- `laila.complex_manifest` holds them as members. Memorizing it is recursive; remembering it triggers **recursive blueprint resolution** (every manifest-scoped leaf comes back as its typed manifest) and `rebuild()` hands back the live `Module` and `Optimizer`.

**Prerequisites:** `pip install "laila-core[s3,torch]"` and a `secrets.toml` with your AWS credentials.

In [ ]:
import torch
import torch.nn as nn
import laila
from laila.data import S3Pool
from laila.addons.torch import Module, Optimizer

In [ ]:
laila.read_args("./secrets.toml")
# secrets.toml example:
#
#   AWS_BUCKET_NAME = "my-bucket"
#   AWS_ACCESS_KEY_ID = "AKIA..."
#   AWS_SECRET_ACCESS_KEY = "..."
#   AWS_REGION = "us-east-1"

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
    nickname="ckpt_pool",
)

laila.memory.extend(s3_pool, pool_nickname="ckpt")

## Step 1: Define a model and optimizer

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

Run a few dummy training steps so the optimizer accumulates state.

In [ ]:
for _ in range(3):
    x = torch.randn(4, 3, 32, 32)
    loss = model(x).sum()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

model.eval()
print("Model trained (dummy steps)")

## Step 2: Wrap the model and the optimizer

`Module` and `Optimizer` behave exactly like the objects they wrap (`forward`, `parameters()`, `step()`, `state_dict()` ...) and are addressable by nickname: the model's checkpoint lives under `MANIFEST:TORCH_MODULE:<nickname>`, the optimizer's under `MANIFEST:TORCH_OPTIMIZER:<nickname>`.

In [ ]:
model = Module(model, nickname="my_cnn")
optimizer = Optimizer(optimizer, nickname="my_cnn_adam")

print(f"model     -> {model.global_id}")
print(f"optimizer -> {optimizer.global_id}")

## Step 3: Put everything under one complex manifest

`everything[key] = value` accepts manifests, manifest-backed objects (the wrappers, or even bare `nn.Module` / `torch.optim.Optimizer` objects), entries and plain blueprint fragments. The blueprint only ever holds gids; the live objects stay attached as *members*.

In [ ]:
everything = laila.complex_manifest(nickname="my_checkpoint")
everything["model"] = model
everything["optimizer"] = optimizer
everything["config"] = laila.constant(data={"lr": 1e-3, "steps": 3, "arch": "SimpleCNN"}, nickname="my_checkpoint/config")

print(f"Manifest global_id: {everything.global_id}")
print(f"Scopes:             {everything._scopes}")
for key in everything.keys():
    print(f"  {key:9s} -> {everything.blueprint[key]}")

## Step 4: Memorize everything to S3

One call. The complex manifest asks each member to memorize itself into the same pool (the optimizer sees its sibling model and records which parameter belongs to which model), batches the plain entries, and finally stores its own blueprint. The returned `GroupFuture` covers every write; `with laila.guarantee:` blocks until all of them are done.

In [ ]:
with laila.guarantee:
    laila.memorize(everything, dst_pool="ckpt")

print(f"Uploaded {sum(1 for _ in everything['model'].manifest)} model entries, "
      f"{sum(1 for _ in everything['optimizer'].manifest)} optimizer entries and the blueprints")

## Step 5: Nuke all local state

Keep a copy of the weights for the final check, then delete every local object. After this, the only way to get the model back is through LAILA.

In [ ]:
reference_state = {name: tensor.clone() for name, tensor in model.state_dict().items()}
reference_optimizer = optimizer.state_dict()

del model, optimizer, everything

print("All local state destroyed")

## Step 6: Reload from the nickname alone

`laila.remember("MANIFEST:COMPLEX:my_checkpoint")` fetches the blueprint; because the scope says `COMPLEX`, LAILA walks it on arrival and casts every manifest-scoped leaf into its typed manifest (**recursive blueprint resolution**) -- the `config` leaf is a plain entry and stays a gid. `rebuild()` then unpickles the architecture skeleton, loads the weights, constructs the optimizer over the rebuilt model's parameters and loads its state.

In [ ]:
with laila.guarantee:
    ref = laila.remember("MANIFEST:COMPLEX:my_checkpoint", dst_pool="ckpt")
everything = ref.result

print(f"Type:    {type(everything).__name__}")
for key in everything.keys():
    print(f"  {key:9s} -> {type(everything[key]).__name__}")

In [ ]:
live = everything.rebuild()
model, optimizer, config = live["model"], live["optimizer"], live["config"].data

print(f"model:     {type(model).__name__} over {type(model.module).__name__}")
print(f"optimizer: {type(optimizer).__name__} over {type(optimizer.optimizer).__name__}")
print(f"config:    {config}")

## Step 7: Verify

Same types, same values: the weights match the copy taken before the nuke, the optimizer state matches, and the optimizer drives the rebuilt model's parameters.

In [ ]:
assert all(torch.equal(reference_state[name], tensor) for name, tensor in model.state_dict().items())
live_ids = {id(p) for p in model.parameters()}
assert all(id(p) in live_ids for group in optimizer.param_groups for p in group["params"])
assert optimizer.state_dict()["param_groups"] == reference_optimizer["param_groups"]

test_input = torch.randn(1, 3, 32, 32)
output = model(test_input)
print(f"Output shape: {output.shape}")
print("Round trip verified")

## Clean up

Forgetting `everything` removes what it owns: the model and optimizer checkpoints (every tensor and their blueprints), the config entry and the complex manifest itself. A plain `Manifest` member (a dataset) would only be dropped as a record -- its entries are shared, never deleted here.

In [ ]:
with laila.guarantee:
    laila.forget(everything, pool="ckpt")

print("Checkpoint cleaned up from S3")

## Summary

- **`Module` / `Optimizer`** (`laila.addons.torch`) wrap the torch objects without changing how you use them; each memorizes as a typed manifest (`MANIFEST:TORCH_MODULE:<nickname>` / `MANIFEST:TORCH_OPTIMIZER:<nickname>`) with one detached CPU entry per tensor.
- **`laila.complex_manifest`** (scope `MANIFEST:COMPLEX`) holds manifests, manifest-backed objects and entries as members; `laila.memorize(everything)` is recursive.
- **Recursive blueprint resolution**: when a complex manifest is remembered, every manifest-scoped leaf is fetched and cast into its typed manifest; plain manifests are never scanned.
- **`everything.rebuild()`** returns `{key: live object}` -- the model rebuilt from its architecture skeleton and weights, the optimizer bound to it with its state loaded, entries and dataset manifests as they are. A round trip gives back the same types and values.
- **`laila.forget(everything)`** is owner-aware: checkpoints are deleted in full, plain manifest members only as records.